In [ ]:

!pip install rasterio joblib xgboost scikit-learn

import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import joblib


In [ ]:
ref_path = "/content/drive/MyDrive/Sikkim/variables/SLOPE_SIKKIM.tif"

with rasterio.open(ref_path) as ref:
    ref_array = ref.read(1)
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_shape = ref_array.shape
    ref_profile = ref.profile

rows, cols = ref_shape
print("Reference shape:", ref_shape)


def resample_to_ref(src_path, ref_shape, ref_transform, ref_crs):
    with rasterio.open(src_path) as src:
        dst_array = np.empty(ref_shape, dtype=np.float32)

        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.nearest
        )

    return dst_array


Reference shape: (3882, 2989)


In [ ]:
raster_paths = [
    "/content/drive/MyDrive/Sikkim/variables/stream_PI.sdat",
    "/content/drive/MyDrive/Sikkim/variables/SLOPE_SIKKIM.tif",
    "/content/drive/MyDrive/Sikkim/variables/Sin_Aspect.tif",
    "/content/drive/MyDrive/Sikkim/variables/ProfileCurvature.tif",
    "/content/drive/MyDrive/Sikkim/variables/ntl_sikkim_30m.tif",
    "/content/drive/MyDrive/Sikkim/variables/Mean_NDVI_Sikkim_2013_2019.tif",
    "/content/drive/MyDrive/Sikkim/variables/final_dis_drainage.tif",
    "/content/drive/MyDrive/Sikkim/variables/dist_tectonic_sikkim_30m.tif",
    "/content/drive/MyDrive/Sikkim/variables/dis_to_Major_roads.tif",
    "/content/drive/MyDrive/Sikkim/variables/lulc_cat.tif",
    "/content/drive/MyDrive/Sikkim/variables/soil_type_cat.tif",
    "/content/drive/MyDrive/Sikkim/variables/lithology_cat.tif"
]


In [ ]:
arrays = []

for path in raster_paths:
    arr = resample_to_ref(
        path,
        ref_shape,
        ref_transform,
        ref_crs
    )
    arrays.append(arr)

stack = np.stack(arrays, axis=-1)
print("Stack shape:", stack.shape)

# Convert invalid values to NaN robustly
stack = stack.astype(np.float32)

# Any extreme negatives or zeros in categorical layers should already be masked
stack[~np.isfinite(stack)] = np.nan

X_map = stack.reshape(-1, stack.shape[-1])
valid_mask = ~np.any(np.isnan(X_map), axis=1)

print("Valid pixel ratio:", valid_mask.sum() / valid_mask.size)


Stack shape: (3882, 2989, 12)
Valid pixel ratio: 0.670032347699766


In [ ]:
# Load bundled model
bundle = joblib.load(
    "/content/drive/MyDrive/LSM_models/xgb_cor2.pkl"
)

model = bundle["model"]
feature_order = bundle["feature_order"]
threshold = bundle["threshold"]

# Sanity check
assert stack.shape[-1] == len(feature_order), \
    "Feature count mismatch!"

y_prob = np.full(X_map.shape[0], np.nan, dtype=np.float32)

y_prob[valid_mask] = model.predict_proba(
    X_map[valid_mask]
)[:, 1]

lsm = y_prob.reshape(rows, cols)


In [ ]:
# --------------------------------------------------
# Invert distance-based features (ML interpretability)
# Higher value = closer = higher susceptibility
# --------------------------------------------------

distance_features = [
    "dis_to_drainage",
    "dist_tectonic",
    "dis_to_roads"
]

for feat in distance_features:
    idx = feature_order.index(feat)

    col = X_map[:, idx]

    # Use valid pixels only for statistics
    col_valid = col[valid_mask]

    dmin = np.nanmin(col_valid)
    dmax = np.nanmax(col_valid)

    if dmax > dmin:
        col_norm = (col - dmin) / (dmax - dmin)
        col_inv = 1.0 - col_norm
    else:
        col_inv = np.zeros_like(col)

    X_map[:, idx] = col_inv

print("Distance sanity check:")
for feat in distance_features:
    idx = feature_order.index(feat)
    print(
        feat,
        "min:", np.nanmin(X_map[valid_mask, idx]),
        "max:", np.nanmax(X_map[valid_mask, idx])
    )

Distance sanity check:
dis_to_drainage min: 0.0 max: 1.0
dist_tectonic min: 0.0 max: 1.0
dis_to_roads min: 0.0 max: 1.0


In [ ]:
# Save feature order & metadata for record
import json

metadata = {
    "model": "XGBoost",
    "features": feature_order,
    "threshold": threshold,
    "crs": ref_crs.to_string(),
    "resolution": ref_transform.a
}

with open("LSM_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("LSM metadata saved.")


LSM metadata saved.


In [ ]:
# Sanity check: inspect a few random pixels
idx = np.where(valid_mask)[0]

sample_idx = np.random.choice(idx, size=5, replace=False)

for i in sample_idx:
    print("Input vector:", X_map[i])
    print("Predicted prob:", model.predict_proba(X_map[i].reshape(1, -1))[0, 1])
    print("-" * 50)


Input vector: [2.5623218e+02 5.6489902e+01 9.9849063e-01 0.0000000e+00 2.4982180e-02
 7.4451947e-01 8.5110754e-01 9.9846315e-01 9.9523044e-01 3.0000000e+00
 3.0000000e+00 1.0000000e+00]
Predicted prob: 0.9448666
--------------------------------------------------
Input vector: [ 2.2608573e+00  4.5984558e+01 -7.1934980e-01 -2.2580387e-04
  1.4258041e-02  6.2706137e-01  8.4743816e-01  9.9862689e-01
  9.3510473e-01  3.0000000e+00  3.0000000e+00  1.0000000e+00]
Predicted prob: 0.982787
--------------------------------------------------
Input vector: [ 5.5166211e+02  3.6731838e+01  6.8743998e-01 -1.6622759e-04
  1.6271086e-02  7.4552971e-01  9.0399027e-01  9.9878502e-01
  9.8567140e-01  3.0000000e+00  4.0000000e+00  2.0000000e+00]
Predicted prob: 0.92640966
--------------------------------------------------
Input vector: [6.1705349e+01 3.4914440e+01 9.4576287e-01 3.9360184e-06 1.4201032e-02
 4.7630203e-01 7.9042155e-01 9.9481660e-01 8.9880782e-01 3.0000000e+00
 3.0000000e+00 1.0000000e+00]
P

In [ ]:
ref_profile.update(
    tiled=True,
    blockxsize=256,
    blockysize=256,
    dtype=rasterio.float32,
    count=1,
    nodata=-9999,
    compress="lzw"
)

lsm_out = lsm.copy()
lsm_out[np.isnan(lsm_out)] = -9999

with rasterio.open("LSM_XGBoost.tif", "w", **ref_profile) as dst:
    dst.write(lsm_out, 1)


In [ ]:
# Flatten valid LSI values
lsi_valid = lsm[~np.isnan(lsm)]

# Quantile thresholds (5 classes)
q = np.percentile(lsi_valid, [20, 40, 60, 80])

print("LSI thresholds:", q)

lsm_class = np.full(lsm.shape, np.nan, dtype=np.float32)

lsm_class[lsm <= q[0]] = 1  # Very Low
lsm_class[(lsm > q[0]) & (lsm <= q[1])] = 2  # Low
lsm_class[(lsm > q[1]) & (lsm <= q[2])] = 3  # Moderate
lsm_class[(lsm > q[2]) & (lsm <= q[3])] = 4  # High
lsm_class[lsm > q[3]] = 5  # Very High

profile_class = ref_profile.copy()
profile_class.update(
    dtype=rasterio.uint8,
    nodata=0,
    count=1,
    compress="lzw"
)

lsm_class_out = np.nan_to_num(lsm_class, nan=0).astype(np.uint8)

with rasterio.open("LSM_XGBoost_Classes.tif", "w", **profile_class) as dst:
    dst.write(lsm_class_out, 1)

print("Classified LSM saved.")



LSI thresholds: [0.04074923 0.11807003 0.24992827 0.49167318]
Classified LSM saved.


In [ ]:
import pandas as pd

unique, counts = np.unique(lsm_class_out, return_counts=True)
stats = dict(zip(unique, counts))

pixel_area_km2 = (abs(ref_transform.a) ** 2) / 1e6

area_stats = {
    k: v * pixel_area_km2
    for k, v in stats.items() if k != 0
}

df_area = pd.DataFrame.from_dict(
    area_stats, orient="index", columns=["Area_km2"]
)

df_area.index = [
    "Very Low", "Low", "Moderate", "High", "Very High"
]

print(df_area)


            Area_km2
Very Low   1399.4253
Low        1399.4253
Moderate   1399.4253
High       1399.4253
Very High  1399.4253
